## Imports and Loading

In [132]:
import pandas as pd
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [134]:
df = pd.read_csv("/Users/jamiewong/Documents/school/2025-26/ECS 111/AI-Hallucinations-Detection/data/cleaned_data.csv")
df.head()

,reference,input,output,label,hallucination_type_realized,question_type,hallucination_type_encouraged
0,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t...","What organization has accredited A.D.A.M., Inc...","A.D.A.M., Inc. is accredited by the Atlantis H...",hallucinated,Entity-error hallucination,Default question type,Entity-error hallucination
1,"""This dataset for NOAA's Science On a Sphere d...",What type of educational activity is encourage...,The dataset encourages learners to complete a ...,hallucinated,Relation-error hallucination,Default question type,Relation-error hallucination
2,"Dimension items include ""Foreground"" and ""Back...",What is the primary reason for a hit to be cla...,The primary reason for a hit to be classified ...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination
3,"Atlas Search runs a new process, called mongot...",What are the specific hardware requirements fo...,"Based on our production monitoring data, mongo...",hallucinated,Unverifiable information hallucination,Other common hallucinated questions,Other hallucination
4,The business implications are stark. In a surv...,What percentage of banking executives in the l...,34% of banking executives in the loan originat...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination


In [8]:
nlp = spacy.load("en_core_web_sm")

## Entity Extraction and Evaluation Functions

In [ ]:
def extract_entities(texts):
    """
    Extract entities for a list of texts using spaCy's nlp.pipe for efficiency.
    Returns a list of dictionaries whose values are sets.
    """
    ner_list = []
    for doc in nlp.pipe(texts, disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"], n_process=-1):
        entities = {}
        for ent in doc.ents:
            entities.setdefault(ent.label_, set()).add(ent.text.lower())
        ner_list.append(entities)
    return ner_list

In [ ]:
def convert_entities(ents):
    """
    converts list of dictionaries into list of sets 
    who contain tuples of the form (TAG, entity)
    """
    ent_list = []
    for item in ents:
        item_ents = [(key, sub_val) for key, values in item.items() for sub_val in values]
        ent_list.append(set(item_ents))
    return ent_list

In [124]:
def calculate_entity_recall(ref_entities, resp_entities):
    """
    ref_entities (set): set of tuples representing the true entities in the reference text
    resp_entities (set): set of tuples representing the predicted entities in the response text
    """
    
    #Spans predicted by the model that exists in the reference text (ground truth)
    tp = len(ref_entities & resp_entities)
    if len(ref_entities) == 0:
        return 1.0

    return tp / len(ref_entities) #context entity recall (RAG)

def out_of_reference_rate(ref_entities, resp_entities):

    if len(resp_entities) == 0:
        return 0.0

    return len(resp_entities - ref_entities) / len(resp_entities) #the response has that amount of entities that are not in the reference

In [ ]:
NUMERIC_LABELS = {
    "CARDINAL",
    "ORDINAL",
    "QUANTITY",
    "MONEY",
    "PERCENT",
    "DATE",
    "TIME",}

def number_overlap(ref_entities, resp_entities):
    """
    input: original list of dictionaries of tags/entities
    """
    ref_nums = set()
    resp_nums = set()

    for label in NUMERIC_LABELS:
        ref_nums.update((label, frozenset(ref_entities.get(label, set()))))
        resp_nums.update((label, frozenset(resp_entities.get(label, set()))))

    if len(ref_nums) == 0:
        return 1.0

    #fraction of numeric entities in the reference that appear in the response
    return len(ref_nums & resp_nums) / len(ref_nums) 

## Running NER and Evaluation

In [ ]:
reference_entities = extract_entities(df['reference'])
response_entities = extract_entities(df['output'])

reference_ents_conv = convert_entities(reference_entities)
response_ents_conv = convert_entities(response_entities)

In [ ]:
ent_recall = []
oor_rate = []
num_overlap = []

for i in range(len(reference_ents_conv)):
    ent_recall.append(calculate_entity_recall(reference_ents_conv[i], response_ents_conv[i]))
    oor_rate.append(out_of_reference_rate(reference_ents_conv[i], response_ents_conv[i]))
    num_overlap.append(number_overlap(reference_entities[i], response_entities[i]))

In [129]:
ner_scores = pd.DataFrame({"Entity Recall": ent_recall,
                           "Out-Of-Reference Rate": oor_rate,
                           "Number Overlap": num_overlap})
ner_scores.head(10)

,Entity Recall,Out-Of-Reference Rate,Number Overlap
0,0.000000,1.000000,1.000000
1,0.000000,1.000000,0.888889
2,0.500000,0.000000,1.000000
3,0.200000,0.900000,1.000000
4,0.333333,0.666667,0.888889
5,0.100000,0.833333,0.888889
6,0.000000,0.000000,1.000000
7,0.500000,0.714286,0.800000
8,0.000000,1.000000,1.000000
9,0.000000,1.000000,1.000000


In [ ]:
#add ref column back for merging later
ner_scores['reference'] = df['reference'] 
ner_scores.head()

,Entity Recall,Out-Of-Reference Rate,Number Overlap,reference
0,0.000000,1.000000,1.000000,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t..."
1,0.000000,1.000000,0.888889,"""This dataset for NOAA's Science On a Sphere d..."
2,0.500000,0.000000,1.000000,"Dimension items include ""Foreground"" and ""Back..."
3,0.200000,0.900000,1.000000,"Atlas Search runs a new process, called mongot..."
4,0.333333,0.666667,0.888889,The business implications are stark. In a surv...


In [142]:
ner_scores.to_csv('NER.csv')